# Audio italiani per il prototipo NLS
Esegui le celle in ordine in Google Colab. Non aggiungere barre inverse alla fine delle righe. Il notebook genera 24 WAV e uno ZIP, senza servizi vocali a pagamento. Il primo avvio scarica modello e voci. Una GPU Colab è facoltativa.

I testi sono una fotografia del prototipo corrente: ascolta e verifica pronunce, numeri e corrispondenza alle procedure prima di integrarli. La generazione non modifica l’app. I file rimangono nella sessione Colab: scarica lo ZIP prima di chiuderla.


In [ ]:
%pip install -q "kokoro>=0.9.2,<1" soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1


In [ ]:
from pathlib import Path
import hashlib, json, shutil, re, importlib.metadata
import numpy as np
import soundfile as sf
import torch
from kokoro import KPipeline
from IPython.display import Audio, display

# Voce italiana: if_sara (femminile) oppure im_nicola (maschile).
VOICE = "if_sara"
SPEED = 0.95
SAMPLE_RATE = 24000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT = Path("/content/nls-audio") / VOICE
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Voce: {VOICE} | dispositivo: {DEVICE}")
pipeline = KPipeline(lang_code="i", repo_id="hexgrad/Kokoro-82M", device=DEVICE)


In [ ]:
AUDIO_SCRIPTS = [{'id': 'preparation', 'category': 'step', 'title': 'Counseling prenatale · Briefing del team · Checklist del materiale', 'text': 'Counseling prenatale. Briefing del team. Checklist del materiale. Partecipa al counseling prenatale e al briefing; chiarisci il compito assegnato e verifica il materiale prima della nascita. Interventi: Counseling prenatale. Briefing del team. Checklist del materiale. Al termine, premi Prosegui.', 'source': 'Poster · Counseling prenatale / briefing / checklist'}, {'id': 'birth', 'category': 'step', 'title': 'Nascita', 'text': 'Nascita. Registra quando è avvenuta la nascita: questo è il riferimento per il tempo trascorso e i target saturazione periferica di ossigeno. Inserisci i minuti trascorsi dalla nascita, poi premi Registra nascita e prosegui.', 'source': 'Poster · Nascita'}, {'id': 'cord', 'category': 'step', 'title': 'Gestione cordone ombelicale', 'text': 'Gestione cordone ombelicale. Concorda il clampaggio con il team e assisti la gestione del cordone. Non ritardare gli interventi necessari. Interventi: Gestione cordone ombelicale. Clampaggio e milking: attendere almeno 60 secondi prima di clampare; considerare milking nei neonati depressi tra 35 e 42 settimane. Non ritardare gli interventi necessari: concorda la gestione con il team. Al termine, premi Prosegui.', 'source': 'Poster · Gestione cordone ombelicale'}, {'id': 'initial', 'category': 'step', 'title': 'Gravidanza a termine? Buon tono? Respira o piange?', 'text': 'Gravidanza a termine? Buon tono? Respira o piange?. Verifica l’età gestazionale, valuta il tono e osserva se il neonato respira o piange. Registra separatamente le tre risposte. Indica separatamente: gravidanza a termine, buon tono, respira o piange. Poi conferma la valutazione.', 'source': 'Poster · Primo bivio'}, {'id': 'skin', 'category': 'step', 'title': 'Skin to skin · Controllo della temperatura · Valutazione continua', 'text': 'Skin to skin. Controllo della temperatura. Valutazione continua. Assisti il contatto skin to skin, controlla la temperatura e continua a valutare il neonato. La valutazione prosegue nel tempo; non si esaurisce con una conferma. Interventi: Skin to skin. Controllo della temperatura. Valutazione continua', 'source': 'Poster · Skin to skin / controllo temperatura / valutazione continua'}, {'id': 'first-actions', 'category': 'step', 'title': 'Fornire calore e mantenere la temperatura · Asciugare · Posizione idonea per la pervietà delle vie aeree · Rimuovere le secrezioni se necessario · Stimolare · Riposizionare', 'text': 'Fornire calore e mantenere la temperatura. Asciugare. Posizione idonea per la pervietà delle vie aeree. Rimuovere le secrezioni se necessario. Stimolare. Riposizionare. Mantieni la temperatura, asciuga e posiziona il neonato per mantenere pervie le vie aeree. Rimuovi le secrezioni solo se necessario, stimola e riposiziona; poi valuta apnea o gasping e frequenza cardiaca. Interventi: Fornire calore e mantenere la temperatura. Asciugare. Posizione idonea per la pervietà delle vie aeree. Rimuovere le secrezioni se necessario. Stimolare. Riposizionare. Al termine, premi Prosegui.', 'source': 'Poster · Primi passi'}, {'id': 'respiration', 'category': 'step', 'title': 'Apnea e/o gasping? FC <100/min?', 'text': 'Apnea o gasping? frequenza cardiaca inferiore a 100 battiti al minuto?. Osserva se sono presenti apnea o gasping e rileva la frequenza cardiaca. Apnea/gasping o frequenza cardiaca inferiore a 100 battiti al minuto richiedono supporto ventilatorio. Inserisci la frequenza cardiaca rilevata in battiti al minuto e indica se sono presenti apnea o gasping. Poi conferma la valutazione.', 'source': 'Poster · Bivio apnea/gasping o FC <100/min'}, {'id': 'distress', 'category': 'step', 'title': 'Respiro difficoltoso e/o cianosi persistente?', 'text': 'Respiro difficoltoso o cianosi persistente?. Osserva il respiro e verifica la presenza di cianosi persistente. Se è presente almeno una delle due condizioni, seleziona SÌ. Indica se sono presenti respiro difficoltoso o cianosi persistente. Poi conferma la valutazione.', 'source': 'Poster · Bivio respiro difficoltoso / cianosi persistente'}, {'id': 'distress-support', 'category': 'step', 'title': 'Posizione idonea per la pervietà delle vie aeree · Monitoraggio SpO₂ · Supplementazione O₂ · Considerare CPAP', 'text': 'Posizione idonea per la pervietà delle vie aeree. Monitoraggio saturazione periferica di ossigeno. Supplementazione ossigeno. Considerare pressione positiva continua nelle vie aeree. Mantieni pervie le vie aeree e attiva il monitoraggio saturazione periferica di ossigeno. Collabora alla supplementazione di ossigeno e valuta con il team la necessità di pressione positiva continua nelle vie aeree. Interventi: Posizione idonea per la pervietà delle vie aeree. Monitoraggio saturazione periferica di ossigeno. Supplementazione ossigeno. Considerare pressione positiva continua nelle vie aeree. Al termine, premi Prosegui.', 'source': 'Poster · Ramo respiro difficoltoso'}, {'id': 'ppv', 'category': 'step', 'title': 'PPV · Monitoraggio SpO₂ · Considerare monitoraggio ECG', 'text': 'ventilazione a pressione positiva. Monitoraggio saturazione periferica di ossigeno. Considerare monitoraggio elettrocardiogramma. Esegui o assisti la ventilazione a pressione positiva. Monitora la saturazione periferica di ossigeno, valuta con il team il monitoraggio elettrocardiogramma e rivaluta la frequenza cardiaca. Interventi: ventilazione a pressione positiva. Monitoraggio saturazione periferica di ossigeno. Considerare monitoraggio elettrocardiogramma. Al termine, premi Prosegui.', 'source': 'Poster · PPV'}, {'id': 'hr100', 'category': 'step', 'title': 'FC <100/min?', 'text': 'frequenza cardiaca inferiore a 100 battiti al minuto?. Rileva la frequenza cardiaca dopo la ventilazione. Se rimane inferiore a 100 battiti al minuto, correggi la ventilazione; altrimenti prosegui con le cure post-rianimatorie. Inserisci la frequenza cardiaca rilevata in battiti al minuto, poi conferma la valutazione.', 'source': 'Poster · Bivio dopo PPV'}, {'id': 'correctives', 'category': 'step', 'title': 'Correttivi della ventilazione · Se necessario tubo tracheale o maschera laringea · Monitor ECG', 'text': 'Correttivi della ventilazione. Se necessario tubo tracheale o maschera laringea. Monitor elettrocardiogramma. Verifica maschera, posizione, secrezioni, apertura della bocca e pressioni. Se necessario prepara una via aerea alternativa e attiva il monitoraggio elettrocardiogramma. Interventi: Correttivi della ventilazione. Se necessario tubo tracheale o maschera laringea. Monitor elettrocardiogramma. Al termine, premi Prosegui.', 'source': 'Poster · Correttivi della ventilazione'}, {'id': 'hr60', 'category': 'step', 'title': 'FC <60/min?', 'text': 'frequenza cardiaca inferiore a 60 battiti al minuto?. Rileva la frequenza cardiaca dopo i correttivi. Prima di avviare le compressioni, conferma con il team che la ventilazione sia efficace. Inserisci la frequenza cardiaca rilevata in battiti al minuto. Prima delle compressioni, conferma con il team la ventilazione efficace. Poi conferma la valutazione.', 'source': 'Poster · Bivio dopo correttivi / messaggi chiave'}, {'id': 'compressions', 'category': 'step', 'title': 'O₂ al 100% · Tubo tracheale o maschera laringea · Compressioni toraciche coordinate con PPV · Monitoraggio ECG · Considerare CVO o IO', 'text': 'ossigeno al 100 per cento. Tubo tracheale o maschera laringea. Compressioni toraciche coordinate con ventilazione a pressione positiva. Monitoraggio elettrocardiogramma. Considerare catetere venoso ombelicale o accesso intraosseo. Collabora all’erogazione di ossigeno al 100 per cento e alla gestione della via aerea. Esegui o assisti le compressioni coordinate con ventilazione a pressione positiva secondo il ruolo assegnato, monitora l’elettrocardiogramma e prepara il materiale per catetere venoso ombelicale o accesso intraosseo se indicato dal team. Interventi: ossigeno al 100 per cento. Tubo tracheale o maschera laringea. Compressioni toraciche coordinate con ventilazione a pressione positiva. Monitoraggio elettrocardiogramma. Considerare catetere venoso ombelicale o accesso intraosseo. Al termine, premi Prosegui.', 'source': 'Poster · Compressioni toraciche coordinate con PPV'}, {'id': 'advanced-hr', 'category': 'step', 'title': 'FC <60/min?', 'text': 'frequenza cardiaca inferiore a 60 battiti al minuto?. Rileva e comunica la frequenza cardiaca dopo le compressioni coordinate con ventilazione a pressione positiva. Registra il valore per verificare se rimane inferiore a 60 battiti al minuto. Inserisci la frequenza cardiaca rilevata in battiti al minuto, poi conferma la valutazione.', 'source': 'Poster · Bivio dopo compressioni'}, {'id': 'medication', 'category': 'step', 'title': 'Adrenalina in CVO o IO ogni 3–5 min · Se FC <60/min persistente considerare ipovolemia · DOPPES', 'text': 'Adrenalina in catetere venoso ombelicale o accesso intraosseo ogni 3 a 5 minuti. Se frequenza cardiaca inferiore a 60 battiti al minuto persistente considerare ipovolemia. Doppes. Verifica la prescrizione e prepara l’adrenalina per la via catetere venoso ombelicale o accesso intraosseo. Registra ogni somministrazione e controlla il tempo trascorso; se frequenza cardiaca rimane inferiore a 60 battiti al minuto, collabora alla valutazione di ipovolemia e delle cause di ventilazione inefficace. Interventi: Adrenalina in catetere venoso ombelicale o accesso intraosseo ogni 3 a 5 minuti. Se frequenza cardiaca inferiore a 60 battiti al minuto persistente, considerare ipovolemia. Rivalutazione Doppes. Al termine, premi Prosegui.', 'source': 'Poster · Adrenalina / ipovolemia / DOPPES'}, {'id': 'recovery-review', 'category': 'step', 'title': 'Rivalutare la risposta del neonato', 'text': 'Rivalutare la risposta del neonato. Comunica la frequenza cardiaca rilevata, rivaluta il respiro e concorda con il team la prosecuzione del supporto. Interventi: Risposta del neonato rivalutata con il team', 'source': 'Limite esplicito della trasposizione del diagramma'}, {'id': 'post', 'category': 'step', 'title': 'Cure post-rianimatorie · Debriefing', 'text': 'Cure post-rianimatorie. Debriefing. Prosegui l’assistenza post-rianimatoria secondo le indicazioni del team e partecipa al debriefing, riferendo valutazioni, interventi e tempi registrati. Interventi: Cure post-rianimatorie. Debriefing del team', 'source': 'Poster · Cure post-rianimatorie / debriefing'}, {'id': 'saturation', 'category': 'reference', 'title': 'Target di saturazione preduttale', 'text': 'Target di saturazione preduttale. A 2 minuti: dal 65 al 70 per cento. A 3 minuti: dal 70 al 75 per cento. A 4 minuti: dal 75 all’80 per cento. A 5 minuti: dall’80 all’85 per cento. A 10 minuti: dall’85 al 95 per cento.', 'source': 'Contenuti del Nido'}, {'id': 'mrsopa', 'category': 'reference', 'title': 'Correttivi MR SOPA', 'text': 'Correttivi della ventilazione in maschera. Maschera adeguata al volto. Riposizionamento in sniffing position. Verifica secrezioni e aspira le vie aeree. Apri (Open) la bocca. Pressioni adeguate. Via aerea alternativa.', 'source': 'Contenuti del Nido'}, {'id': 'doppes', 'category': 'reference', 'title': 'Verifiche DOPPES', 'text': 'Cause di ventilazione inefficace nel neonato intubato. Dislocazione. Ostruzione. Pressioni adeguate. Pneumotorace. Strumentazione (Equipment) ben funzionante. Rimuovere aria dallo stomaco.', 'source': 'Contenuti del Nido'}, {'id': 'newborn', 'category': 'interface', 'title': 'Dati dichiarati nello scenario', 'text': 'Inserisci, se disponibili, l’età gestazionale in settimane compiute e la temperatura rilevata in gradi Celsius. Poi premi Registra dati dello scenario.', 'source': 'Contenuti del Nido'}, {'id': 'help', 'category': 'interface', 'title': 'Aiuto e coordinamento del team', 'text': 'Attiva il sistema di chiamata previsto dalla struttura e condividi il passaggio attuale. Nessuna chiamata viene effettuata dall’app. Conferma sullo schermo la richiesta effettuata nello scenario.', 'source': 'Contenuti del Nido'}, {'id': 'finished', 'category': 'interface', 'title': 'Simulazione conclusa', 'text': 'Simulazione conclusa. Puoi esportare il registro o avviare un’altra sessione. Il registro corrente viene perso chiudendo o ricaricando la pagina.', 'source': 'Contenuti del Nido'}]
print(f"{len(AUDIO_SCRIPTS)} audio: 18 passaggi, 3 supporti, 3 messaggi di interfaccia.")


In [ ]:
# Genera un WAV per passaggio. I segmenti restituiti da Kokoro vengono uniti.
# Rilanciare questa cella riprende i file completati, se testo e impostazioni coincidono.
manifest = {"voice": VOICE, "speed": SPEED, "sample_rate": SAMPLE_RATE,
            "model": "hexgrad/Kokoro-82M", "kokoro_version": importlib.metadata.version("kokoro"),
            "purpose": "Prototipo per simulazione; audio da ascoltare e verificare", "clips": []}
failures = []
for index, clip in enumerate(AUDIO_SCRIPTS, 1):
    path = OUTPUT / f"{clip['id']}.wav"
    text_path = OUTPUT / f"{clip['id']}.txt"
    stamp = OUTPUT / f"{clip['id']}.sha256"
    fingerprint = hashlib.sha256(json.dumps({"text": clip["text"], "voice": VOICE,
        "speed": SPEED, "model": manifest["model"], "version": manifest["kokoro_version"]},
        sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    try:
        cached = path.exists() and stamp.exists() and stamp.read_text() == fingerprint
        if cached:
            audio, sr = sf.read(path, dtype="float32")
            if sr != SAMPLE_RATE or not audio.size or not np.isfinite(audio).all():
                raise ValueError("File esistente non valido: elimina il WAV e riprova.")
        else:
            chunks = []
            for _, _, segment in pipeline(clip["text"], voice=VOICE, speed=SPEED, split_pattern=r"\n+"):
                if segment is None:
                    continue
                if hasattr(segment, "detach"):
                    segment = segment.detach().cpu().numpy()
                segment = np.asarray(segment, dtype=np.float32).reshape(-1)
                if segment.size:
                    chunks.append(segment)
            if not chunks:
                raise ValueError("Il modello non ha prodotto audio.")
            audio = np.concatenate(chunks)
            if not np.isfinite(audio).all():
                raise ValueError("Campioni audio non validi.")
            sf.write(path, audio, SAMPLE_RATE, subtype="PCM_16")
            stamp.write_text(fingerprint)
        text_path.write_text(clip["text"], encoding="utf-8")
        manifest["clips"].append({**clip, "file": path.name,
            "duration_seconds": round(len(audio) / SAMPLE_RATE, 2), "sha256_text_settings": fingerprint})
        print(f"[{index}/{len(AUDIO_SCRIPTS)}] {path.name}: {len(audio)/SAMPLE_RATE:.1f} s" + (" (già pronto)" if cached else ""))
    except Exception as exc:
        failures.append({"id": clip["id"], "error": str(exc)})
        print(f"ERRORE {clip['id']}: {exc}")

(OUTPUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT / "errors.json").write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")
if failures:
    raise RuntimeError(f"{len(failures)} audio non generati. Correggi gli errori e rilancia questa cella. ZIP non creato.")
assert len(manifest["clips"]) == len(AUDIO_SCRIPTS)
archive = shutil.make_archive(str(OUTPUT.parent / f"nls-audio-{VOICE}"), "zip", root_dir=OUTPUT)
print(f"\nCompletati {len(manifest['clips'])} audio. Durata totale: {sum(c['duration_seconds'] for c in manifest['clips'])/60:.1f} minuti")
# Anteprima di una richiesta di parametri; gli altri file sono nello ZIP.
display(Audio(filename=str(OUTPUT / "respiration.wav")))
from google.colab import files
files.download(archive)
